# Primer XGBoost: conteos y alerta semanal

Este notebook ejecuta `src.modeling.train`. Usa el gold validado para h=2 y h=4, compara cuatro frente a seis variables sobre las mismas filas y entrena regresión de `log1p(casos_Dengue)` y clasificación directa de `brote`. La regresión produce también una alerta al aplicar el umbral histórico de la fila. La persistencia es la referencia.

Validación: temporadas 2021–2023. Prueba principal: temporada 2024. Prueba adicional del cambio de fuente: año calendario 2025 completo. Se reentrena antes del primer origen de cada temporada y se fija el umbral de probabilidad solo con la validación.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.modeling.train import ejecutar_experimento

reporte = ejecutar_experimento()

h=2: 30160 filas modelables; 325 de arranque fuera de la matriz
  historia_4, temporada_2021: train=11960, test=3445, corte=2020-08-15
  historia_4, temporada_2022: train=15405, test=3380, corte=2021-08-21
  historia_4, temporada_2023: train=18785, test=3380, corte=2022-08-20
  historia_4, temporada_2024: train=22165, test=3380, corte=2023-08-19
  historia_4, calendario_2025: train=26715, test=3380, corte=2024-12-21
  historia_4_mas_2, temporada_2021: train=11960, test=3445, corte=2020-08-15
  historia_4_mas_2, temporada_2022: train=15405, test=3380, corte=2021-08-21
  historia_4_mas_2, temporada_2023: train=18785, test=3380, corte=2022-08-20
  historia_4_mas_2, temporada_2024: train=22165, test=3380, corte=2023-08-19
  historia_4_mas_2, calendario_2025: train=26715, test=3380, corte=2024-12-21
h=4: 30030 filas modelables; 455 de arranque fuera de la matriz
  historia_4, temporada_2021: train=11700, test=3445, corte=2020-08-01
  historia_4, temporada_2022: train=15145, test=3380, corte

In [2]:
import pandas as pd

filas = []
for h, bloque in reporte["horizontes"].items():
    for variante, experimento in bloque["variantes"].items():
        for fold in experimento["folds"]:
            if not fold["tipo"].startswith("prueba"):
                continue
            for enfoque, valores in (("regresion", fold["regresion"]["alerta_derivada"]),
                                      ("clasificacion", fold["clasificacion"]["alerta_directa"]),
                                      ("persistencia", fold["persistencia"]["alerta_derivada"])):
                filas.append({"h": int(h), "variante": variante, "bloque": fold["bloque"],
                              "enfoque": enfoque, "positivos": valores["positivos"],
                              "VP": valores["vp"], "FP": valores["fp"],
                              "precision": valores["precision"], "recall": valores["recall"],
                              "F1": valores["f1"], "AUPRC": valores["auprc"]})
resumen = pd.DataFrame(filas)
display(resumen.sort_values(["h", "bloque", "variante", "enfoque"]))

,h,variante,bloque,enfoque,positivos,VP,FP,precision,recall,F1,AUPRC
4,2,historia_4,calendario_2025,clasificacion,3,1,241,0.004132,0.333333,0.008163,0.008203
5,2,historia_4,calendario_2025,persistencia,3,0,8,0.000000,0.000000,0.000000,0.010223
3,2,historia_4,calendario_2025,regresion,3,0,3,0.000000,0.000000,0.000000,0.033780
10,2,historia_4_mas_2,calendario_2025,clasificacion,3,2,320,0.006211,0.666667,0.012308,0.009147
11,2,historia_4_mas_2,calendario_2025,persistencia,3,0,8,0.000000,0.000000,0.000000,0.010223
9,2,historia_4_mas_2,calendario_2025,regresion,3,0,3,0.000000,0.000000,0.000000,0.033689
1,2,historia_4,temporada_2024,clasificacion,857,754,708,0.515732,0.879813,0.650280,0.663073
2,2,historia_4,temporada_2024,persistencia,857,690,183,0.790378,0.805134,0.797688,0.842831
0,2,historia_4,temporada_2024,regresion,857,653,124,0.840412,0.761960,0.799266,0.865573
7,2,historia_4_mas_2,temporada_2024,clasificacion,857,811,881,0.479314,0.946324,0.636328,0.668587


El año calendario 2025 tiene solo tres positivos, por lo que sus métricas de alerta son muy inestables y se reportan como sensibilidad separada. No escoger el enfoque con las métricas de prueba: la decisión operativa sobre falsos avisos y casos perdidos sigue pendiente de discusión. Los modelos, predicciones y métricas quedan en `models/experimentos/` y `docs/modeling/metricas/`.